# Head retraining with language-stratified CV

Retrains the `LNMLPHead` on ESD with speaker-grouped k-fold CV — on English-only,
Chinese-only, or all speakers (set `LANGUAGE`). Architecture: frozen wav2vec2 ->
mean-pool of the last hidden state -> LNMLPHead (input LayerNorm, dropout 0.1).

**Output** (in `OUT_DIR`): `results_<LANGUAGE>.json`, `history_fold*.json`,
`curves_<LANGUAGE>.png`.

All training logic lives in `retrain_lib.py`; this notebook only sets config and
calls it. Requires the repo (`common/` + `4_model_retraining/`) on the Colab path.

In [ ]:
!pip install -q soundfile librosa resampy transformers

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports & repo path

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Repo root on Colab: must contain common/ and 4_model_retraining/ (git clone or Drive sync).
REPO_ROOT = Path("/content/drive/MyDrive/ser_andreaingoglia_unimi")
sys.path.insert(0, str(REPO_ROOT / "4_model_retraining"))

import retrain_lib as R
from retrain_lib import TrainConfig

device = R.set_seed(42)
print(f"Device: {device}")

## 3. Configuration

In [ ]:
# LANGUAGE: "english", "chinese", or "all"
LANGUAGE = "all"

# Paths (Colab / Drive)
ESD_GT_CSV     = Path("/content/drive/MyDrive/ESD/ESD_GT_Colabo.csv")
ESD_ZIP_PATH   = Path("/content/drive/MyDrive/ESD_ZIP.zip")
ESD_AUDIO_ROOT = Path("/content/drive/MyDrive/ESD")
LOCAL_CACHE    = Path("/content/audio_cache_esd")
OUT_DIR        = Path(f"/content/drive/MyDrive/head_retrain_{LANGUAGE}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
cfg = TrainConfig(num_epochs=15, dropout=0.1, balance_classes=True)

print(f"LANGUAGE={LANGUAGE} | dropout={cfg.dropout} | folds={N_FOLDS} | out={OUT_DIR}")

## 4. Load ESD audio + ground truth

In [ ]:
cache = R.prepare_local_cache(ESD_ZIP_PATH, LOCAL_CACHE, ESD_AUDIO_ROOT)
path_map = R.build_path_map(cache)
print(f"Path map: {len(path_map)} files")

esd_df = R.load_esd_gt(ESD_GT_CSV, path_map)
if LANGUAGE != "all":
    esd_df = esd_df[esd_df["language"] == LANGUAGE].reset_index(drop=True)
if len(esd_df) == 0:
    raise RuntimeError(f"No samples for language='{LANGUAGE}'")

LABELS = R.SHARED_EMOTIONS
print(f"Samples: {len(esd_df)} | speakers: {esd_df['speaker'].nunique()} "
      f"| languages: {esd_df['language'].value_counts().to_dict()}")
print(esd_df["emotion"].value_counts())

## 5. Speaker-grouped CV splits

In [ ]:
# Stratify folds by language when training on all speakers; plain k-fold otherwise.
if LANGUAGE == "all":
    splits = R.split_speakers_kfold_stratified(esd_df, N_FOLDS, seed=cfg.seed)
else:
    speakers = sorted(esd_df["speaker"].unique())
    splits = R.split_speakers_kfold(speakers, N_FOLDS, seed=cfg.seed)

for i, (tr, va) in enumerate(splits, 1):
    n_tr = len(esd_df[esd_df["speaker"].isin(tr)])
    n_va = len(esd_df[esd_df["speaker"].isin(va)])
    print(f"  Fold {i}: train {len(tr)} spk ({n_tr}) | val {len(va)} spk ({n_va}) -> {va}")

## 6. Cross-validation training

In [ ]:
fold_results = []
for fold_idx, (train_spk, val_spk) in enumerate(splits, start=1):
    print(f"\n{'='*60}\n  Fold {fold_idx}/{N_FOLDS} | LANGUAGE={LANGUAGE}")
    print(f"  Val speakers: {val_spk}\n{'='*60}")

    res = R.train_one(esd_df, train_spk, val_spk, LABELS, cfg, device)
    bm = res["best_metrics"]
    print(f"  Fold {fold_idx} best: acc={res['best_val_acc']:.4f} "
          f"f1={bm.get('val_f1', 0):.4f} @epoch {res['best_epoch']}")

    fold_results.append({
        "fold": fold_idx, "train_speakers": train_spk, "val_speakers": val_spk,
        "best_val_acc": res["best_val_acc"], "best_val_f1": bm.get("val_f1", 0),
        "best_val_prec": bm.get("val_prec", 0), "best_val_rec": bm.get("val_rec", 0),
        "best_epoch": res["best_epoch"], "best_cm": bm.get("confusion_matrix", []),
        "best_per_class_recall": bm.get("per_class_recall", {}),
        "epochs_trained": len(res["history"]), "history": res["history"],
    })

print("\nAll folds complete!")

## 7. Summary

In [ ]:
accs  = [r["best_val_acc"]  for r in fold_results]
f1s   = [r["best_val_f1"]   for r in fold_results]
precs = [r["best_val_prec"] for r in fold_results]
recs  = [r["best_val_rec"]  for r in fold_results]

print(f"RESULTS - LANGUAGE={LANGUAGE}, dropout={cfg.dropout}")
print(f"  {'Fold':<6s}|{'Acc':>8s}|{'F1':>8s}|{'Prec':>8s}|{'Rec':>8s}|{'Epoch':>6s}")
for r in fold_results:
    print(f"  {r['fold']:<6d}|{r['best_val_acc']:>8.4f}|{r['best_val_f1']:>8.4f}|"
          f"{r['best_val_prec']:>8.4f}|{r['best_val_rec']:>8.4f}|{r['best_epoch']:>6d}")
print(f"  {'Mean':<6s}|{np.mean(accs):>8.4f}|{np.mean(f1s):>8.4f}|"
      f"{np.mean(precs):>8.4f}|{np.mean(recs):>8.4f}|")
print(f"  {'Std':<6s}|{np.std(accs):>8.4f}|{np.std(f1s):>8.4f}|"
      f"{np.std(precs):>8.4f}|{np.std(recs):>8.4f}|")

print("\nPer-class recall (mean +/- std across folds):")
for emo in LABELS:
    vals = [r["best_per_class_recall"].get(emo, 0) for r in fold_results]
    print(f"  {emo:<12s}: {np.mean(vals):.3f} +/- {np.std(vals):.3f}")

## 8. Save results

In [ ]:
report = {
    "language": LANGUAGE, "dropout": cfg.dropout, "input_layernorm": True,
    "n_folds": N_FOLDS, "num_epochs": cfg.num_epochs,
    "mean_acc": float(np.mean(accs)), "std_acc": float(np.std(accs)),
    "mean_f1": float(np.mean(f1s)),  "std_f1": float(np.std(f1s)),
    "per_fold": [{k: v for k, v in r.items() if k != "history"} for r in fold_results],
}
with open(OUT_DIR / f"results_{LANGUAGE}.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
for r in fold_results:
    with open(OUT_DIR / f"history_fold{r['fold']}.json", "w") as f:
        json.dump(r["history"], f, indent=2)
print(f"Saved results + histories to {OUT_DIR}")

## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Head retraining - {LANGUAGE} - dropout={cfg.dropout}", fontsize=13)
for r in fold_results:
    h = r["history"]; eps = [e["epoch"] for e in h]; f = r["fold"]
    axes[0].plot(eps, [e["train_acc"] for e in h], label=f"F{f}", alpha=0.7)
    axes[0].plot(eps, [e["val_acc"] for e in h], "--", label=f"F{f} val", alpha=0.7)
    axes[1].plot(eps, [e["train_loss"] for e in h], label=f"F{f}", alpha=0.7)
    axes[1].plot(eps, [e["val_loss"] for e in h], "--", label=f"F{f} val", alpha=0.7)
    axes[2].plot(eps, [e["val_f1"] for e in h], label=f"F{f}", alpha=0.7)
for ax, t in zip(axes, ["Accuracy", "Loss", "Val F1"]):
    ax.set_title(t); ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT_DIR / f"curves_{LANGUAGE}.png", dpi=150)
plt.show()
print(f"Curves saved to {OUT_DIR}")